#### Status: WIP.
Preliminary analysis on the first tracer batch. Provides the high-level metrics cited in the README. Detailed error-mode analysis in progress — observed failure modes include routing errors and premature SBERT resolution where LLM escalation was warranted (SBERT threshold tuning pending).

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
from pathlib import Path
import json
from pprint import pprint
import notebook_support.notebook_config as nb_cfg
from game_app.types import SessionAggregates, SessionReport, PerformanceMetrics
from game_app.metrics_aggregator import calculate_performance_metrics


In [2]:
# load the legacy and synthetic tracer datasets
filename = "tracer_production_green_v1.parquet"

path = nb_cfg.FINAL_DATA_DIR / filename
table =  pq.read_table(path)
df = table.to_pandas()
df.shape

(139, 33)

In [6]:
## create session aggregate DTO list

telemetry_data_path = nb_cfg.DATA_DIR / "11_runtime_session_reports_metrics/vm/game_data"
report_files = sorted(telemetry_data_path.glob("session_aggregates_*.json"))

# sanity check — how many files matched
print(f"{len(report_files)} session aggregate files found in {telemetry_data_path.name}")  

all_session_aggregates = []

for f in report_files:
    with open(f, "r", encoding="utf-8")  as file:
        data = json.load(file)
    # multiple sessions within a game -> list of dicts in json    
    if isinstance(data, list):
        all_session_aggregates.extend(SessionAggregates(**item) for item in data)
    # single session -> dict in json    
    else:
        all_session_aggregates.append(SessionAggregates(**data))
        
print(f"{len(all_session_aggregates)} session aggregates created.")    

29 session aggregate files found in game_data
80 session aggregates created.


In [7]:
## Filter out session aggregates with no executed questions (ie. user quit before answering any questions)

filtered_reports = [
    r for r in all_session_aggregates
    if not (r.session_quit is True and r.executed_question_count == 0)
]

print(f"Filtered reports: {len(filtered_reports)} / {len(all_session_aggregates)}")

Filtered reports: 74 / 80


In [13]:
## calculate performance metrics across session aggregate

performance_metrics = calculate_performance_metrics(scope_id="tracer_batch_1_telemetry",
                                                       session_aggregates=all_session_aggregates, 
                                                       scope="batch")

pprint(performance_metrics.model_dump())

{'accuracy_rate': 72.46621621621621,
 'average_evaluation_latency': 2.5645380067567567,
 'average_llm_call_latency': 2.3754824482161423,
 'correct_answer_count': 429.0,
 'executed_questions': 592,
 'llm_routing_share': 29.222972972972972,
 'max_evaluation_latency': 40.2988,
 'max_llm_call_latency': 33.991580963134766,
 'min_evaluation_latency': 0.0,
 'min_llm_call_latency': 0.0,
 'num_sessions': 80,
 'overall_tier_distribution': {'exact': 167,
                               'fuzzy': 45,
                               'llm': 173,
                               'sbert': 196,
                               'unresolved': 11},
 'p95_evaluation_latency': 8.5003,
 'p95_llm_call_latency': 9.10171973705291,
 'quit_rate': 11.25,
 'sbert_routing_share': 33.108108108108105,
 'scope': 'batch',
 'scope_id': 'tracer_batch_1_telemetry',
 'shift_left_resolution_rate': 68.91891891891892,
 'tier_distribution_by_evaluator': {'EX': {'empty_submission': 2,
                                           'ex_exac

Player accuracy (proportion of questions answered correctly): a proxy for question difficulty, not a measure of evaluator correctness. Evaluator accuracy against labelled answers is measured separately (see BoD Quality constraint).

In [14]:
# open production dataset (does not have tensors)
# load the legacy and synthetic tracer datasets
filename = "tracer_production_green_v1.parquet"
path = nb_cfg.FINAL_DATA_DIR / filename
runtime_table =  pq.read_table(path)
df = runtime_table.to_pandas()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139 entries, 0 to 138
Data columns (total 33 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   mcq_options                                 45 non-null     object 
 1   question_type                               139 non-null    object 
 2   question_source                             139 non-null    object 
 3   question                                    139 non-null    object 
 4   answer                                      139 non-null    object 
 5   answer_variations                           139 non-null    object 
 6   hint_1                                      139 non-null    object 
 7   hint_2                                      139 non-null    object 
 8   hint_3                                      139 non-null    object 
 9   explanation                                 139 non-null    object 
 10  semantic_entit

In [16]:
df.columns

Index(['mcq_options', 'question_type', 'question_source', 'question', 'answer',
       'answer_variations', 'hint_1', 'hint_2', 'hint_3', 'explanation',
       'semantic_entity_refs', 'semantic_lore_concepts', 'master_id',
       'question_embeddings', 'answer_embeddings',
       'answer_variations_embeddings', 'source_quote_embeddings',
       'original_question_id', 'syn_id', 'source_reference', 'source_quote',
       'mcq_distractors_embeddings', 'question_length', 'answer_length',
       'answer_type', 'question_tokens', 'answer_tokens',
       'combined_unique_tokens', 'main_keyword', 'question_embeddings_tensor',
       'answer_embeddings_tensor',
       'answer_variations_embeddings_tensor_matrix',
       'mcq_distractors_embeddings_tensor_matrix'],
      dtype='object')

In [22]:

question = "Why did Lord Voldemort order Nagini to kill Severus Snape in the Shrieking Shack during the battle of Hogwarts?"
keywords = ["nagini", "severus", "snape", "shrieking", "shack", "battle", "hogwarts"]

mask = df['answer_tokens'].apply(
    lambda tokens: any(token in keywords for token in tokens)
)
df.loc[(mask)&(df['main_keyword']=="why"), ["master_id", "question", "answer"]]

,master_id,question,answer
109,FFBjZ64Q,Why did Lord Voldemort order Nagini to kill Se...,voldemort mistakenly believed that because sna...
110,Z6wdYgLD,Why did Albus Dumbledore explicitly ask Severu...,dumbledore wanted to protect draco malfoy's so...
111,sTn1jfG4,"During the Battle of Hogwarts, why did Hermion...",hermione understood that nagini was lord volde...
113,1rn0irvx,Why was Lord Voldemort certain that Harry Pott...,"voldemort believed that harry's ""one great fla..."
